# TorchSigGeoDataset Example with Mobile Objects

This notebook demonstrates the **TorchSigGeoDataset** for simulating RF propagation from multiple transmitters to multiple receivers with realistic channel effects including **path loss** and **path delay**.

## Overview

The TorchSig geo module enables the creation of synthetic geolocation datasets by:
- Placing **transmitters** at specific geographic coordinates (latitude, longitude, altitude)
- Placing **receivers** at other geographic coordinates
- Simulating **RF propagation effects** between each transmitter-receiver pair, including:
  - **Path loss**: Signal attenuation based on distance and propagation model
  - **Path delay**: Time delay based on distance (speed of light propagation)
- Generating **IQ signal data** at each receiver that combines signals from all visible transmitters

This is useful for:
- Training and testing **geolocation algorithms** (TDOA, FDOA, RSSI-based)
- Simulating **wireless network scenarios** with realistic RF propagation
- Generating **synthetic data** for radio frequency machine learning applications

---

## Key Concepts

### Geographic Coordinates
- **GeoPoint**: Represents a physical location with latitude, longitude, and altitude
- **Transmitter**: A signal source at a specific GeoPoint location
- **Receiver**: A signal observer at a specific GeoPoint location

### Propagation Modeling
- **PathLoss**: Models signal attenuation over distance (free-space or custom loss)
- **PathDelay**: Models propagation time delay (distance / speed of light)
- Both transforms use the actual geographic distance between transmitter and receiver

### Dataset Structure
- Each receiver produces one sample containing the combined signal from all transmitters
- Component signals are stored separately for ground truth analysis
- Metadata includes transmitter/receiver positions, path loss values, and delays

## Setup and Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Import TorchSig components
from torchsig.datasets.datasets import TorchSigIterableDataset
from torchsig.geo.types import GeoPoint
from torchsig.geo.datasets import TorchSigGeoDataset, Transmitter, Receiver
from torchsig.geo.transforms import PathLoss, PathDelay
from torchsig.geo.utils.file_handler import GeoDatasetReader
from torchsig.utils.defaults import TorchSigDefaults
from torchsig.transforms.transforms import Spectrogram

# Import example utility functions for visualization and geolocation
from examples.scripts.geo_example_utils import (
    plot_geo_network_at_frame,
    collect_multi_frame_data,
    estimate_position_from_ranges,
    calculate_tdoa_fix,
)

## 1. Create Dataset Metadata

Customize signal generation parameters for the geo scenario.

In [ ]:
# Get default metadata and customize for geo example
defaults = TorchSigDefaults()
dataset_metadata = defaults.default_dataset_metadata.copy()

# Use a smaller dataset size for faster generation in this example
dataset_metadata["num_iq_samples_dataset"] = 65536  # 2^16 samples
dataset_metadata["fft_size"] = 256
dataset_metadata["fft_stride"] = 256
dataset_metadata["sample_rate"] = 10_000_000  # 10 MHz

# Use positive frequencies that fit within the sample rate's Nyquist limit
dataset_metadata["frequency_min"] = 0
dataset_metadata["frequency_max"] = 2_500_000
dataset_metadata["signal_center_freq_min"] = 0
dataset_metadata["signal_center_freq_max"] = 2_500_000

## 2. Define Position Generators for Mobile Objects

Create helper functions to generate positions. Center point is San Francisco (37.7749, -122.4194).

In [ ]:
# Center point (San Francisco)
center_lat = 37.7749
center_lon = -122.4194

# Standard deviation for normal distribution (in degrees)
# 0.001 degrees = ~100-125 meters spread
position_std = 0.001
step_std = 0.00025  # Movement per frame (~25-30 meters)

from typing import Callable

def create_moving_position_func(center_lat: float, center_lon: float,
                                position_std: float, step_std: float,
                                seed: int = None) -> Callable[[int], GeoPoint]:
    """Create a callable that returns positions for mobile objects (true random walk).
    
    Each frame's position is computed deterministically from the frame index.
    Same frame_index always returns same GeoPoint.
    """
    import numpy as np
    
    def position_func(frame_index: int) -> GeoPoint:
        deterministic_seed = seed if seed is not None else 0
        init_rng = np.random.default_rng(deterministic_seed)
        
        lat = init_rng.normal(center_lat, position_std)
        lon = init_rng.normal(center_lon, position_std)
        
        # True random walk: accumulate steps
        for f in range(1, frame_index + 1):
            step_rng = np.random.default_rng(deterministic_seed + f)
            lat += step_rng.normal(0, step_std)
            lon += step_rng.normal(0, step_std)
        
        return GeoPoint(lat=lat, lon=lon, alt=50.0)
    
    return position_func

def create_circular_position_func(center_lat: float, center_lon: float,
                                  radius_deg: float, speed: float,
                                  seed: int = None) -> Callable[[int], GeoPoint]:
    """Create a callable that returns positions on a circular path."""
    import numpy as np
    deterministic_seed = seed if seed is not None else 0
    rng = np.random.default_rng(deterministic_seed)
    initial_angle = rng.uniform(0, 2 * np.pi)
    
    def position_func(frame_index: int) -> GeoPoint:
        angle = initial_angle + speed * frame_index
        return GeoPoint(
            lat=center_lat + radius_deg * np.cos(angle),
            lon=center_lon + radius_deg * np.sin(angle),
            alt=50.0
        )
    
    return position_func

def create_fixed_position_func(lat: float, lon: float, alt: float) -> Callable[[int], GeoPoint]:
    """Create a callable that returns a fixed position."""
    def position_func(frame_index: int) -> GeoPoint:
        return GeoPoint(lat=lat, lon=lon, alt=alt)
    return position_func

# Number of frames to simulate
num_frames = 50

# TX-1: Circular path (~100m radius) inside receiver triangle
# TX-2: Fixed high-altitude (~2km) ~2.8km northwest
tx_position_funcs = [
    create_circular_position_func(center_lat, center_lon, 0.0009, 0.1, seed=1),
    create_fixed_position_func(center_lat + 0.025, center_lon - 0.025, 2000.0),
]

# Signal types for each transmitter
tx_signal_types = ["bpsk", "qpsk"]

# 3 receivers at corners of ~2km triangle with small deterministic motion
rx_position_funcs = [
    create_moving_position_func(center_lat + 0.018, center_lon, 0.0002, 0.00005, seed=10),
    create_moving_position_func(center_lat - 0.009, center_lon + 0.015, 0.0002, 0.00005, seed=11),
    create_moving_position_func(center_lat - 0.009, center_lon - 0.015, 0.0002, 0.00005, seed=12),
]

print("Position functions created")
for i, pf in enumerate(tx_position_funcs):
    pos = pf(0)
    print(f"  TX-{i+1}: ({pos.lat:.6f}, {pos.lon:.6f}, {pos.alt:.1f}m)")
for i, pf in enumerate(rx_position_funcs):
    pos = pf(0)
    print(f"  RX-{i+1}: ({pos.lat:.6f}, {pos.lon:.6f}, {pos.alt:.1f}m)")

## 3. Create Transmitters

Each transmitter wraps a TorchSigIterableDataset with a geographic position.

### Transmitter Architecture

A `Transmitter` combines:
- A `TorchSigIterableDataset` that generates synthetic signals
- A **callable position function** defining where the transmitter moves over frames
- An identifier string for tracking

Each transmitter generates signals independently with its own seed for reproducibility.

### Two Transmitters in This Example

- **TX-1 (BPSK)**: Main transmitter on a **circular path** (~100m radius) inside the receiver triangle - this is the target for geolocation demonstration
- **TX-2 (QPSK)**: Interfering transmitter at a **fixed position** ~2.8 km northwest - this adds interference at all receivers but is NOT geolocated

### Note on Determinism
All position functions are **deterministic**: calling `get_position(frame_index)` multiple times for the same `frame_index` will always return the same `GeoPoint`. This ensures reproducible simulation results.

In [ ]:
# Signal types for each transmitter
# tx_1: Main transmitter (BPSK) - target for geolocation
# tx_2: Interfering transmitter (QPSK) - adds interference at receivers
tx_signal_types = ["bpsk", "qpsk"]

# Create transmitters with distinct frequency bands
# Note: Frequencies are OFFSETS from dataset origin (0 Hz), not absolute RF frequencies
transmitters = []
for i, (pos_func, sig_type) in enumerate(zip(tx_position_funcs, tx_signal_types)):
    tx_metadata = dataset_metadata.copy()
    
    if i == 0:  # TX-1 (BPSK): 250-750 kHz offset
        tx_metadata["signal_center_freq_min"] = 250_000
        tx_metadata["signal_center_freq_max"] = 750_000
        tx_metadata["frequency_min"] = 0
        tx_metadata["frequency_max"] = 1_000_000
    else:  # TX-2 (QPSK): 1.75-2.25 MHz offset
        tx_metadata["signal_center_freq_min"] = 1_750_000
        tx_metadata["signal_center_freq_max"] = 2_250_000
        tx_metadata["frequency_min"] = 1_500_000
        tx_metadata["frequency_max"] = 2_500_000

    tx_dataset = TorchSigIterableDataset(
        metadata=tx_metadata,
        signal_generators=[sig_type],
        seed=i * 1000,
    )

    transmitter = Transmitter(
        dataset=tx_dataset,
        position=pos_func,
        identifier=f"tx_{i+1}"
    )
    transmitters.append(transmitter)

print("Transmitters created")
for tx in transmitters:
    print(f"  {tx}")

print("\nTransmitter positions (first 3 frames):")
for frame_idx in range(min(3, num_frames)):
    print(f"  Frame {frame_idx}:")
    for tx in transmitters:
        pos = tx.get_position(frame_idx)
        print(f"    {tx.identifier}: ({pos.lat:.6f}, {pos.lon:.6f})")

## 4. Create Receivers

Receivers are defined by their geographic positions.

### Receiver Architecture

A `Receiver` is simpler than a transmitter:
- A **callable position function** defining where the receiver moves over frames
- An identifier string for tracking
- Optional receiver-specific transforms

Each receiver will capture the combined signal from all transmitters, with each
transmitter's signal attenuated and delayed based on the **current** propagation path.

### Receiver Placement Strategy

The three receivers are positioned at the corners of a ~2km equilateral triangle for 
excellent TDOA geometry. TX-1 moves in a ~100m circle at the center, well inside the 
receiver triangle. Each receiver has small deterministic motion (~5-6m per frame) to 
simulate realistic movement.

In [ ]:
# 3 receivers at corners of ~2km equilateral triangle with small deterministic motion
receivers = []
for i, pos_func in enumerate(rx_position_funcs):
    receiver = Receiver(
        position=pos_func,
        sample_rate=dataset_metadata["sample_rate"],
        identifier=f"rx_{i+1}"
    )
    receivers.append(receiver)

print("Receivers created with small deterministic motion:")
for rx in receivers:
    if callable(rx._position):
        print(f"  {rx} (mobile)")
    else:
        print(f"  {rx} (static)")

# Show positions for first few frames (receivers move slightly each frame)
print("\nReceiver positions across frames:")
for frame_idx in range(min(5, num_frames)):
    print(f"\n  Frame {frame_idx}:")
    for rx in receivers:
        pos = rx.get_position(frame_idx)
        print(f"    {rx.identifier}: ({pos.lat:.6f}, {pos.lon:.6f}, {pos.alt:.1f}m)")

## 5. Plot Network Topology at Multiple Frames

We'll use matplotlib to create a simple 2D map showing the transmitter and receiver locations
at different frames.

### Visualizing the Network Topology

This visualization helps understand:
- The spatial relationship between transmitters (red triangles) and receivers (blue circles)
- How positions change over frames
- The connection paths between each transmitter-receiver pair
- **TX-1** (inside the receiver triangle) - the target for TDOA geolocation
- **TX-2** (far northwest) - the interferer at a distance

In [ ]:
# plot_geo_network_at_frame is now imported from geo_example_utils

# Plot first 4 frames in a 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
frame_indices = [0, 15, 30, 45]

for idx, (ax, frame_idx) in enumerate(zip(axes.flat, frame_indices)):
    plot_geo_network_at_frame(transmitters, receivers, frame_idx, connections=True, ax=ax)

plt.suptitle('Network Topology Across Frames (TX-1 Inside Triangle, TX-2 Far Northwest with Distinct Frequency Bands)', fontsize=16, y=0.98)
plt.tight_layout()
plt.show()

## 6. Apply Channel Transforms (Path Loss and Path Delay)

`PathLoss` simulates signal attenuation over distance. `PathDelay` simulates propagation time delays. For mobile objects, these use the current transmitter-receiver distance at each frame.

Both transforms are essential for realistic propagation modeling.

In [ ]:
# Channel transforms

# Path loss transform (free space propagation model)
path_loss = PathLoss(model="free_space")

# Path delay transform (uses speed of light by default)
path_delay = PathDelay()

# Combine into channel transforms list
channel_transforms = [path_delay, path_loss]

print("Channel transforms:")
for t in channel_transforms:
    print(f"  {t}")

## 7. Create TorchSigGeoDataset and Collect Multi-Frame Data

Now we create the main geo dataset with our transmitters, receivers, and channel transforms.

### How TorchSigGeoDataset Works with Mobile Objects

The `TorchSigGeoDataset` orchestrates the simulation by:

1. **Iterating over receivers**: Each sample corresponds to one receiver's perspective
2. **Computing frame_index**: `frame_index = receiver_counter // len(receivers)`
3. **For each receiver, processing all transmitters**:
   - Get transmitter position at current frame_index via `get_position()`
   - Get receiver position at current frame_index via `get_position()`
   - Generate a signal from each transmitter's dataset
   - Apply channel transforms based on the **current** transmitter-receiver distance
   - Store the transformed signal as a component signal
4. **Combining component signals**: All transmitter signals are summed
5. **Adding metadata**: Receiver position, transmitter info, path loss, delays at current frame

### Collecting Multi-Frame Data

Since the dataset iterates sequentially through receivers, we need to collect multiple
iterations to get data from multiple frames. Each complete pass through all receivers
gives us one frame of data.

### Two Transmitters Scenario with Distinct Positive Frequency Bands

With 2 transmitters, each receiver sample will contain:
- The combined IQ signal from BOTH transmitters (TX-1 + TX-2)
- **TX-1 (BPSK)**: Lower positive frequency band (0-1 MHz), inside receiver triangle
- **TX-2 (QPSK)**: Upper positive frequency band (1.5-2.5 MHz), far northwest
- Component signals for each transmitter separately (for ground truth analysis)
- Path loss and delay metadata for each transmitter-receiver path
- This illustrates how the spectrograms aggregate signals at the receiver
- The **frequency separation** (0.5 MHz gap) makes the two transmitters visually distinct on spectrograms

In [ ]:
RF_CENTER_FREQ = 2.4e9

# Create the geo dataset with mobile objects
geo_dataset = TorchSigGeoDataset(
    transmitters=transmitters,
    receivers=receivers,
    channel_transforms=channel_transforms,
    center_freq=RF_CENTER_FREQ,
)

# Collect multi-frame data
all_frames = collect_multi_frame_data(geo_dataset, num_frames=num_frames)

# Create topology for visualization
from torchsig.geo.utils.coordinate_system import lla_to_ecef
from torchsig.geo.types import GeoPoint

# Build topology dictionary for the first frame
topology = {'paths': {}}
for tx in transmitters:
    for rx in receivers:
        tx_pos = tx.get_position(0)
        rx_pos = rx.get_position(0)
        distance = tx_pos.distance_to(rx_pos)
        topology['paths'][(tx.identifier, rx.identifier)] = {
            'transmitter': tx,
            'receiver': rx,
            'distance_m': distance
        }

print("\nPath distances (Frame 0):")
for (tx_id, rx_id), info in topology['paths'].items():
    print(f"  {tx_id} -> {rx_id}: {info['distance_m']/1000:.2f} km")

# Note: With 2 transmitters, TX-1 is at the center (~1.6-2.1km from each receiver)
# and TX-2 is ~2.8 km northwest of center (~3.0-3.5 km from each receiver).
# The path loss for TX-2 will be significantly higher due to the longer distance,
# making it a weaker interferer at the receivers.

## 8. Inspect Multi-Frame Data

Let's inspect the collected multi-frame data and see how positions and delays change.

### Understanding the Sample Structure

Each sample from the geo dataset contains:

**Main signal data**:
- `data`: Complex IQ samples representing the **combined signal** from ALL transmitters at the receiver
- `data.shape`: (num_samples,) for complex data

**Receiver metadata**:
- `rx_id`: Identifier for the receiver
- `rx_lat`, `rx_lon`, `rx_alt`: Receiver position at this frame
- `frame_index`: Which frame this sample is from
- `receiver_index`: Which receiver in the frame
- `num_transmitters`: Number of transmitters contributing to this sample
- `tx_ids`: List of transmitter identifiers

**Component signals**:
- `component_signals`: List of individual transmitter signals **before combining**
- Each component has its own `data`, metadata (including `tx_id`, `class_name`)
- Each component also has `path_loss_db` and `path_delay_seconds` metadata
- This allows you to see each transmitter's contribution separately

### Key Insight: Signal Aggregation at Receiver

The `data` field contains the sum of all transmitter signals after applying path loss and delay.
The `component_signals` field contains each transmitter's signal individually. This is useful for:
- Understanding how signals combine at the receiver
- Analyzing interference patterns
- Training classifiers on individual transmitter signals within a mixed signal

In [ ]:
# Inspect first frame, first receiver
first_frame = all_frames[0]
frame_0_rx_0 = first_frame[0]

print(f"Sample type: {type(frame_0_rx_0)}")
print(f"Sample data shape: {frame_0_rx_0.data.shape}")
print(f"Sample data dtype: {frame_0_rx_0.data.dtype}")
print(f"\nSample metadata:")
for key in ['rx_id', 'rx_lat', 'rx_lon', 'rx_alt', 'frame_index', 'receiver_index', 'num_transmitters', 'tx_ids']:
    if key in frame_0_rx_0.keys():
        print(f"  {key}: {frame_0_rx_0[key]}")

print(f"\nNumber of component signals: {len(frame_0_rx_0.component_signals)}")
for i, comp in enumerate(frame_0_rx_0.component_signals):
    actual_signal = comp.component_signals[0] if comp.component_signals else comp
    tx_id = comp['tx_id']
    class_name = actual_signal['class_name']
    print(f"  Component {i}: tx_id={tx_id}, shape={comp.data.shape}, class_name={class_name}")
    if 'path_loss_db' in comp.keys():
        print(f"    Path loss: {comp['path_loss_db']:.2f} dB")
    if 'path_delay_seconds' in comp.keys():
        print(f"    Path delay: {comp['path_delay_seconds']*1e6:.2f} us")

# Show how positions change across frames
print("\nPosition changes across frames:")
for frame_idx in range(min(5, len(all_frames))):
    print(f"\n  Frame {frame_idx}:")
    for rx_idx, sample in enumerate(all_frames[frame_idx]):
        print(f"    RX-{rx_idx+1}: ({sample['rx_lat']:.6f}, {sample['rx_lon']:.6f})")
        for comp in sample.component_signals:
            tx_id = comp['tx_id']
            delay_us = comp.component_signals[0]['path_delay_seconds'] * 1e6
            loss_db = comp.component_signals[0]['path_loss_db']
            print(f"      {tx_id}: delay={delay_us:.2f} us, loss={loss_db:.2f} dB")

# Compare signal power: TX-1 (close) vs TX-2 (far)
print("\n--- Signal Power Comparison (Frame 0, RX-1) ---")
first_sample = all_frames[0][0]
for comp in first_sample.component_signals:
    tx_id = comp['tx_id']
    signal_power = np.mean(np.abs(comp.data)**2)
    path_loss = comp.component_signals[0]['path_loss_db']
    print(f"  {tx_id}: signal_power={signal_power:.2e}, path_loss={path_loss:.2f} dB")
    
# The combined signal power should be dominated by TX-1 (less path loss)
combined_power = np.mean(np.abs(first_sample.data)**2)
print(f"  Combined: signal_power={combined_power:.2e}")
print("  Note: TX-1 (BPSK, close) has much higher power than TX-2 (QPSK, far)")
print("  Both signals are present in the combined receiver signal!")

## 9. Plot Signals - Spectrogram View

Create side-by-side spectrogram plots showing transmitter signals (left) and the combined receiver signal (right).

### Spectrogram Comparison with Two Transmitters

**Left column (Transmitters)**: Individual transmitter signals (BPSK and QPSK) before propagation effects.

**Right column (Receivers)**: Combined signal at each receiver after applying path loss and delay.

Key observations with 2 transmitters:
- **TX-1 (BPSK, close, lower positive frequency band 0-1 MHz)**: Stronger signal with less path loss (~104-106 dB), appears in the lower frequency portion of the spectrogram
- **TX-2 (QPSK, far, upper positive frequency band 1.5-2.5 MHz)**: Weaker signal with more path loss (~112-114 dB), appears in the upper frequency portion of the spectrogram
- The receiver spectrogram shows **both signals combined** - the aggregation of signals at the receiver
- The BPSK signal (TX-1) will be more visible due to higher power AND lower frequency band
- The QPSK signal (TX-2) will be fainter but still present in the upper band, illustrating interference
- This demonstrates how the spectrograms aggregate signals from multiple transmitters at the receiver
- The **0.5 MHz frequency gap** between bands makes it easy to visually distinguish the two transmitters on the spectrogram

In [ ]:
# Create side-by-side layout: transmitters on left, all receivers on right
import matplotlib.gridspec as gridspec

# Use the already-collected multi-frame data (frame 0) which has consistent
# transmitter signals across all receivers. This ensures all receivers are
# listening to the SAME transmitter broadcasts, not independent simulations.
first_frame = all_frames[0]  # All receivers from frame 0
first_sample = first_frame[0]
n_tx = len(first_sample.component_signals)
n_rx = len(first_frame)

# Get fft_size and fft_stride from metadata
fft_size = dataset_metadata.get("fft_size", 256)
fft_stride = dataset_metadata.get("fft_stride", 256)

# Create figure with constrained_layout to handle colorbars properly
# With 2 transmitters and 3 receivers, we need enough vertical space
fig = plt.figure(figsize=(14, 4 * max(n_tx, n_rx)), constrained_layout=True)
gs = gridspec.GridSpec(max(n_tx, n_rx), 2, width_ratios=[1, 1], wspace=0.4, hspace=0.5)

# Plot transmitter spectrograms (left column)
# All receivers share the same transmitter signals from frame 0
for i, comp_signal in enumerate(first_sample.component_signals):
    actual_signal = comp_signal.component_signals[0] if comp_signal.component_signals else comp_signal
    tx_id = comp_signal['tx_id']
    class_name = actual_signal['class_name']

    tx_spectrogram = Spectrogram(fft_size=fft_size)(comp_signal.copy())
    ax = fig.add_subplot(gs[i, 0])
    im = ax.imshow(tx_spectrogram.data, aspect='auto', cmap='viridis')
    ax.set_title(f'{tx_id} ({class_name}) - Transmitter', pad=10)
    ax.set_xlabel('Time')
    ax.set_ylabel('Frequency')
    fig.colorbar(im, ax=ax)

# Turn off remaining left column subplots if n_rx > n_tx
for i in range(n_tx, max(n_tx, n_rx)):
    ax = fig.add_subplot(gs[i, 0])
    ax.axis('off')

# Plot receiver spectrograms (right column) - all from the SAME frame
# Now all receivers show the COMBINED signal from BOTH transmitters
# This illustrates how spectrograms aggregate signals at the receiver
for i, sample in enumerate(first_frame):
    rx_spectrogram = Spectrogram(fft_size=fft_size)(sample.copy())
    ax = fig.add_subplot(gs[i, 1])
    im = ax.imshow(rx_spectrogram.data, aspect='auto', cmap='viridis')
    ax.set_title(f"{sample['rx_id']} - Receiver (Combined: TX-1 0-1MHz + TX-2 1.5-2.5MHz)", pad=10)
    ax.set_xlabel('Time')
    ax.set_ylabel('Frequency')
    fig.colorbar(im, ax=ax)

# Turn off remaining right column subplots if n_tx > n_rx
for i in range(n_rx, max(n_tx, n_rx)):
    ax = fig.add_subplot(gs[i, 1])
    ax.axis('off')

plt.suptitle('Spectrogram View: Individual Transmitters (Left - Separate Positive Frequency Bands) vs Combined Receiver Signal (Right)', fontsize=16, y=0.98)
plt.show()

## 10. Write Data to Disk Using .yaml/.dat Format

Save the geo dataset to disk using the .yaml (metadata) and .dat (IQ data) file pair format.

### Geo Dataset File Format

The geo dataset uses a paired file format for efficient storage:

- **`.yaml` files**: Contain metadata in YAML format (receiver position, transmitter info, path loss, delay, etc.)
- **`.dat` files**: Contain raw IQ data in binary format (float32 by default)

Each receiver gets its own file pair: `rx_<id>_<index>.yaml` and `rx_<id>_<index>.dat`

### Field Mapping

The `field_mapping` parameter controls how metadata fields are named in the output files. This allows customization for compatibility with existing tools or conventions.

### Dataset Info Files

Additional files created:
- `dataset_info.yaml`: Overall dataset metadata and configuration
- `writer_info.yaml`: Information about the writing process

These files help reconstruct the dataset when reading back from disk.

In [ ]:
# Create output directory
output_dir = Path('./datasets/geo_example')
output_dir.mkdir(parents=True, exist_ok=True)

# Re-create the geo dataset for writing (since iterator was exhausted)
geo_dataset = TorchSigGeoDataset(
    transmitters=transmitters,
    receivers=receivers,
    channel_transforms=channel_transforms,
    center_freq=RF_CENTER_FREQ,
)

# Write dataset to .yaml/.dat files
# We'll write a small number of samples for demonstration
dataset_length = 6  # 2 samples per receiver

print(f"Writing {dataset_length} samples to {output_dir}...")

# Use to_yaml_dat_pairs for .yaml/.dat format
geo_dataset.to_yaml_dat_pairs(
    root=str(output_dir),
    dataset_length=dataset_length,
    data_type='float32',
    field_mapping={
        'rx_lat': 'lat',
        'rx_lon': 'lon', 
        'rx_alt': 'alt',
        'tx_lat': 'tx_lat',
        'tx_lon': 'tx_lon',
        'tx_alt': 'tx_alt',
    },
    overwrite=True,
    multithreading=False,  # Disable for reproducibility
)

print(f"Done! Files written to {output_dir}")

# List the files created
print(f"\nFiles created:")
for f in sorted(output_dir.glob('*')):
    print(f"  {f.name}")

## 11. Read Data from Disk

Load the saved .yaml/.dat files back into Signal objects.

### Using GeoDatasetReader

The `GeoDatasetReader` provides efficient reading of geo dataset files:
- Automatically pairs .yaml metadata with .dat IQ data
- Supports the same field mapping used during writing
- Provides random access to samples by index
- Returns Signal objects with full metadata

This allows you to save a geo dataset once and reload it multiple times without regenerating the synthetic data.

In [ ]:
# Create a reader
reader = GeoDatasetReader(
    root=str(output_dir),
    field_mapping={
        'lat': 'rx_lat',
        'lon': 'rx_lon',
        'alt': 'rx_alt',
    }
)

print(f"Reader created. Dataset length: {len(reader)}")

# Read and display the first few samples
print(f"\nReading samples from disk:")
for i in range(min(3, len(reader))):
    signal = reader.read(i)
    print(f"\n  Sample {i}:")
    print(f"    rx_id: {signal['rx_id']}")
    print(f"    Position: ({signal['rx_lat']}, {signal['rx_lon']})")
    print(f"    Data shape: {signal.data.shape}")
    print(f"    Data dtype: {signal.data.dtype}")
    print(f"    Transmitters: {signal['tx_ids']}")

## 12. Range-Based Geolocation with Multi-Frame Averaging

Demonstrate **range-based multilateration** geolocation for **TX-1 only** (the BPSK transmitter
on a circular path) using the ground-truth propagation delays from the simulation metadata.

### Important Methodology Note

This example uses **range-based multilateration** (also called multilateration or 
range-based localization), **NOT** true Time Difference of Arrival (TDOA). Here's the difference:

- **Range-based multilateration** (what this example does): Uses **absolute propagation delays** 
  (time = distance / speed_of_light) from transmitter to each receiver. Requires synchronized 
  clocks between transmitter and receivers, or knowledge of the absolute transmit time.
  
- **True TDOA**: Uses **differences in arrival times** between receiver pairs. The unknown 
  transmit time cancels out when taking differences, so clock synchronization is not required.

In this simulation, we have access to the ground-truth `path_delay_seconds` metadata for 
each transmitter-receiver path. In a real system, you would need to estimate these delays 
from the received signal (e.g., via cross-correlation with a known waveform), and the 
accuracy would be affected by interference from TX-2.

### Multi-Frame Averaging with a Moving Target

Since TX-1 is moving on a circular path (~100m radius), each frame has a different true 
position. The plots below demonstrate the multilateration approach across the trajectory.

In [ ]:
# Perform range-based multilateration for TX-1 and plot results

# Re-use the already collected all_frames data
tx_id = "tx_1"

avg_lat, avg_lon, avg_alt, frame_estimates = calculate_tdoa_fix(
    all_frames, transmitter_idx=0, transmitters=transmitters
)

# Get TX-1 positions for each frame (for circular path visualization)
tx1_true_positions = [transmitters[0].get_position(i) for i in range(len(all_frames))]

# Plot multilateration fixes for TX-1
fig, ax = plt.subplots(figsize=(10, 8))

# Plot true trajectory (circular path)
true_lons = [p.lon for p in tx1_true_positions]
true_lats = [p.lat for p in tx1_true_positions]
ax.plot(true_lons, true_lats, 'r-', linewidth=2, label='TX-1 Trajectory (Circular)', zorder=5)

# Plot average position from multilateration
if avg_lat is not None:
    ax.scatter(avg_lon, avg_lat, c='green', marker='x', s=200, label='Multilateration Average', zorder=5)
    ax.text(avg_lon + 0.0005, avg_lat - 0.0005, 'Multilateration Avg', fontsize=10, color='green')

# Plot per-frame true positions (small dots)
ax.scatter(true_lons, true_lats, c='red', marker='o', s=30, alpha=0.5, label='TX-1 Positions', zorder=4)

# Plot per-frame estimates (small dots)
if frame_estimates:
    frame_lons = [e[1] for e in frame_estimates]
    frame_lats = [e[0] for e in frame_estimates]
    ax.scatter(frame_lons, frame_lats, c='blue', marker='o', s=30, alpha=0.5, label='Frame Estimates')
    # Connect estimates to show estimation trajectory
    ax.plot(frame_lons, frame_lats, 'b--', linewidth=1, alpha=0.5)

# Plot TX-2 position (interferer, not being located) in a different color
tx2_lat = np.mean([p.lat for p in [transmitters[1].get_position(i) for i in range(len(all_frames))]])
tx2_lon = np.mean([p.lon for p in [transmitters[1].get_position(i) for i in range(len(all_frames))]])
ax.scatter(tx2_lon, tx2_lat, c='purple', marker='s', s=200, label='TX-2 (Interferer)', zorder=5)
ax.text(tx2_lon + 0.0005, tx2_lat + 0.0005, 'TX-2 Interferer', fontsize=10, color='purple')

# Plot receiver positions for context
for rx in receivers:
    pos = rx.get_position(0)
    ax.scatter(pos.lon, pos.lat, c='cyan', marker='o', s=100, alpha=0.7, label='Receivers' if rx.identifier == 'rx_1' else "")
    ax.text(pos.lon + 0.0004, pos.lat - 0.0004, rx.identifier, fontsize=9, color='cyan')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'{tx_id} Range-Based Multilateration')
ax.legend(fontsize=8, loc='best')
ax.grid(True, alpha=0.3)
ax.axis('equal')

plt.suptitle('Range-Based Multilateration: TX-1 on Circular Path with TX-2 Interferer', fontsize=16)
plt.tight_layout()
plt.show()

print(f"\n{tx_id} Range-Based Multilateration:")
print(f"  Number of frame estimates: {len(frame_estimates) if frame_estimates else 0}")
if avg_lat is not None:
    print(f"  Average estimated position: ({avg_lat:.6f}, {avg_lon:.6f}, {avg_alt:.1f}m)")

## Summary

This notebook has demonstrated the complete workflow for creating and using TorchSigGeoDataset
with **mobile objects** and **multiple transmitters** for synthetic RF signal data generation.

### Workflow Recap

1. **Dataset Metadata**: Customized signal generation parameters for geo applications
2. **Position Generators**: Created deterministic callable functions for transmitter/receiver positions:
   - TX-1: Circular path (~100m radius) using `create_circular_position_func`
   - TX-2: Fixed position ~2.8 km northwest using `create_fixed_position_func`
   - Receivers: Small deterministic random walk around nominal positions using `create_moving_position_func`
3. **Network Topology**: 2 transmitters (TX-1: BPSK on circular path, TX-2: QPSK fixed) and 3 receivers at ~2km triangle
4. **Transmitters**: Different signal types (BPSK, QPSK) at moving/fixed positions with **separate frequency offset bands**
5. **Receivers**: Positioned at corners of a ~2km triangle for excellent multilateration geometry
6. **Channel Transforms**: Applied PathLoss and PathDelay for realistic RF propagation
7. **Geo Dataset**: Created TorchSigGeoDataset to orchestrate the simulation
8. **Multi-Frame Collection**: Collected data from 50 frames for analysis
9. **Signal Inspection**: Examined signal aggregation at receivers
10. **Spectrogram Visualization**: Compared individual transmitter spectrograms with combined receiver spectrograms
11. **Time Domain Visualization**: Showed signal aggregation in time domain
12. **Range-Based Multilateration**: Demonstrated position estimation using ground-truth propagation delays

### Key Clarifications

- **Determinism**: All position functions are deterministic - same frame_index always returns same position
- **Frequency Offsets**: All frequencies are **offsets from the dataset origin (0 Hz)**, not absolute RF frequencies:
  - TX-1 (BPSK): 250-750 kHz offset range
  - TX-2 (QPSK): 1.75-2.25 MHz offset range (0.5 MHz gap for visual separation)
- **Range-Based Multilateration vs TDOA**: This example uses **range-based multilateration** with absolute delays, not true TDOA which uses delay differences. In practice:
  - Range-based: Requires synchronized clocks or known transmit time. Accuracy depends on absolute delay estimation.
  - True TDOA: Uses delay differences between receiver pairs. No clock synchronization required.
- **Ground-Truth vs Realistic**: The geolocation demonstration uses ground-truth metadata (`path_delay_seconds`) directly, not estimates extracted from the combined IQ signal. In a real system with interference from TX-2, you would need to:
  - Extract arrival times from the combined receiver signal (e.g., via cross-correlation)
  - Separate overlapping signals (e.g., using known waveforms or blind source separation)
  - Account for TX-2 interference which would degrade estimation accuracy
- **Moving Target**: TX-1 follows a circular path. The multilateration visualization shows how frame estimates follow the trajectory.

### Geometry Notes

- TX-1 to receivers: ~1.6-2.1 km (at center of triangle)
- TX-2 to receivers: ~3.0-3.5 km (northwest of center)
- Receiver triangle baseline: ~2 km
- This geometry provides sub-100m localization accuracy for a transmitter inside the triangle

### When to Use This Framework

- Training and testing **geolocation algorithms** with controlled scenarios
- Simulating **wireless network scenarios** with realistic RF propagation
- Generating **synthetic data** with known ground truth for ML research
- Testing **interference robustness** with multiple simultaneous transmitters
- Exploring **mobility effects** on signal characteristics